# HDF5 trials
Trials to save the dataset as hdf5 with OME-XML specifications

In [ ]:
from pathlib import Path
from dateutil import parser
from ome_types.model import OME, Image, Pixels, UnitsLength, PixelType
from imageio.v3 import immeta

In [ ]:
# Extract datetime from metadata
def parse_date(metadata: dict) -> str:
    """Parse date as ISO format from image metadata"""
    date_str = metadata["User"]["Date"]
    time_str = metadata["User"]["Time"]
    datetime_str = f"{date_str} {time_str}"
    acquisition_date = parser.parse(datetime_str)
    return acquisition_date.isoformat()

In [ ]:
tileset_path = Path("../data/raw/Au_01-vol_01")

excluded_file = tileset_path / "excluded_tiles.txt"
excluded_tiles = excluded_file.read_text().splitlines()

tile_list = [
    path for path in tileset_path.glob("*.tif") if path.name not in excluded_tiles
]

# Read metadata: size information should be the same in all images
metadata = immeta(tile_list[0])

In [ ]:
ome = OME()
image = Image(
    id=tileset_path.name,
    acquisition_date=parse_date(metadata),
    pixels=Pixels(
        dimension_order="XYZCT",
        size_x=metadata["Image"]["ResolutionX"],
        size_y=metadata["Image"]["ResolutionY"],
        size_z=len(tile_list),
        size_c=1,
        size_t=1,
        physical_size_x=metadata["Scan"]["PixelWidth"],
        physical_size_x_unit=UnitsLength.METER,
        physical_size_y=metadata["Scan"]["PixelHeight"],
        physical_size_y_unit=UnitsLength.METER,
        physical_size_z=50e-9,
        physical_size_z_unit=UnitsLength.METER,
        type=PixelType.UINT8,
    ),
)

/Users/boda/Library/Caches/pypoetry/virtualenvs/sphero-vem-S4wgO9hY-py3.12/lib/python3.12/site-packages/pydantic/main.py:214: UserWarning: Casting invalid ImageID 'Au_01-vol_01' to 'Image:2'
  validated_self = self.__pydantic_validator__.validate_python(data, self_instance=self)


## To do
- Validate XML
- Save image as hdf5
- Check if channel order is correctly read!
- Check if fiji/napari can open the files as well 